In [1]:
import mysql.connector

#Modify the connection to include the database name
database_name="Code_with_tim_test_database"
db=mysql.connector.connect(
    host="localhost",
    user="root",
    password="gpt67rSy",
    database=database_name, # <-- We are now connecting to the new database
    buffered=True,# best practice - for parallel multiple queries,prevents error,standard practice
)
if db.is_connected():
    print(f"Successfully connected to the '{db.database}' database.")

#We can now create cursor to work within this database
my_cursor=db.cursor()

# You can now execute queries that affect this database,
# like creating tables, inserting data, etc.

Successfully connected to the 'code_with_tim_test_database' database.


In [2]:
my_cursor.execute("SHOW TABLES")
for row in my_cursor:
    print(row)

('person',)
('test',)


#### Topic 2: Example Relationship (One-to-One)

**Explanation:**
We will create two tables: `users` and `scores`. This is a **one-to-one** relationship, meaning each user has exactly one set of scores, and each score entry belongs to exactly one user. In this case, the `user_id` in the `scores` table is both a primary key and a foreign key.

#### 1. Create the Parent Table (`users`)

In [3]:
# Query to create users table (run this first) # practical way
Q1="""
CREATE TABLE IF NOT EXISTS users(
    id INT AUTO_INCREMENT NOT NULL,
    name VARCHAR(50),
    passw VARCHAR(50),
    PRIMARY KEY(id)
)
"""
my_cursor.execute(Q1)

### 2. Create the Child Table (`scores`) with Foreign Key

In [4]:
# Query to create scores table (run this after users table is created) # practical way
Q2="""
CREATE TABLE scores(
    user_id INT,
    PRIMARY KEY(user_id),
    FOREIGN KEY(user_id) REFERENCES users(id),
    game1 INT DEFAULT 0,
    game2 INT DEFAULT 0
)
"""
my_cursor.execute(Q2)

In [5]:
# for checking tables in ur databases
my_cursor.execute("SHOW TABLES")
for row in my_cursor:
    print(row)

('person',)
('scores',)
('test',)
('users',)


#### Topic 3: Inserting Data and Establishing the Relationship

**Explanation:**
We will insert users into the `users` table. For each user we create, we will immediately add their corresponding scores into the `scores` table. We use `cursor.lastrowid` to get the newly generated `id` of the user, which we then use as the foreign key (`user_id`) in the `scores` table.

In [9]:
users_data=[
    ("Kristal","pass123"),
    ("Biraj","pass456"),
    ("Tim","pass789")
]

user_scores=[
    (45,100),
    (32,200),
    (15,300)
]

#### Insert users and their scores

In [11]:
insert_user_query="INSERT INTO users (name,passw) VALUES (%s,%s)"
insert_score_query="INSERT INTO scores (user_id,game1,game2) VALUES (%s,%s,%s)"
for i,user in enumerate(users_data):
    my_cursor.execute(insert_user_query,user)
    last_user_id=my_cursor.lastrowid #get the id of the newly inserted user
    my_cursor.execute(insert_score_query,(last_user_id,user_scores[i][0],user_scores[i][1]))

db.commit() #commit all changes to the database


In [14]:
my_cursor.execute("SHOW TABLES")
for row in my_cursor:
    print(row)

('person',)
('scores',)
('test',)
('users',)


In [16]:
print("users table \n")
my_cursor.execute("SELECT * from users")
for row in my_cursor:
    print(row)
print("SCORES TABLE\n")
my_cursor.execute("select * from scores")
for row in my_cursor:
    print(row)

users table 

(1, 'Kristal', 'pass123')
(2, 'Biraj', 'pass456')
(3, 'Tim', 'pass789')
(4, 'Kristal', 'pass123')
(5, 'Biraj', 'pass456')
(6, 'Tim', 'pass789')
SCORES TABLE

(1, 45, 100)
(2, 32, 200)
(3, 15, 300)
(4, 45, 100)
(5, 32, 200)
(6, 15, 300)
